In [6]:
import os
import json
import re
from dotenv import load_dotenv
load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
QLOO_API_KEY = os.getenv("QLOO_API_KEY")

In [2]:
# core/llm_parser.py
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

from src.custom_exception import CustomException
from src.logger import get_logger

logger = get_logger("llm_parser")

async def extract_preferences(mensaje: str, model: str, api_key: str) -> dict:
    try:
        logger.info("##### INITIALIZING LLM PARSER.PY ##### ")

        try:
                logger.info("##### Creating RAG chain #####")
                prompt = """
You are a cultural assistant that prepares audience intelligence queries using the Qloo API.

From the user's input, extract a structured profile of the intended audience using this format:

{{
  "filter_type": "urn:tag",
  "take": 10,
  "signals": {{
    "demographics": {{
      "age": "35_and_younger"
    }},
    "location": {{
      "query": ["New York", "Los Angeles"]
    }},
    "entities": {{
      "urn:entity:artist": ["Radiohead"],
      "urn:entity:book": ["Norwegian Wood"],
      "urn:entity:movie": ["Pulp Fiction"],
      "urn:entity:tv_show": [],
      "urn:entity:video_game": [],
      "urn:entity:destination": ["Tokyo"]
    }}
  }}
}}

Guidelines:
- Fill in the "entities" with items the user clearly references.
- If the user does not mention a type (e.g., no books), leave that list empty.
- If age or location is missing, do not guess.
- Always return a valid JSON (no markdown, no explanations).

User input:
{document}
"""
                llm = ChatOpenAI(
                    base_url="https://openrouter.ai/api/v1",
                    openai_api_key=api_key,
                    model=model
                )
                prompt_template = ChatPromptTemplate.from_template(prompt)
                logger.info("##### FINISHED - Creating RAG chain #####")
        except Exception as e:
                logger.error(f"Error while Creating RAG chain: {e}")
                raise CustomException("Error while Creating RAG chain", e)
        try:
                logger.info("##### GETTING THE ANSWER FROM CHAIN #####")
                answer = prompt_template | llm | StrOutputParser()
                final_answer = answer.invoke({"document": mensaje})
                logger.info("##### FINISHED - GETTING THE ANSWER FROM CHAIN #####")
        except Exception as e:
                logger.error(f"Error while GETTING THE ANSWER FROM CHAIN: {e}")
                raise CustomException("Error GETTING THE ANSWER FROM CHAIN", e)
        
        logger.info("##### FINISHED -- INITIALIZING LLM PARSER.PY ##### ")
        return final_answer
    except Exception as e:
        logger.error(f"ERROR INITIALIZING LLM PARSER.PY: {e}")
        raise CustomException("ERROR INITIALIZING LLM PARSER.PY", e)




In [5]:
parsed_response = await extract_preferences("I'm developing a series for young women in New York who love experimental film and indie music.", "deepseek/deepseek-chat-v3-0324:free", OPENROUTER_API_KEY)

In [9]:
parsed_response

'```json\n{\n  "filter_type": "urn:tag",\n  "take": 10,\n  "signals": {\n    "demographics": {\n      "age": "35_and_younger",\n      "gender": "female"\n    },\n    "location": {\n      "query": ["New York"]\n    },\n    "entities": {\n      "urn:entity:artist": [],\n      "urn:entity:book": [],\n      "urn:entity:movie": [],\n      "urn:entity:tv_show": [],\n      "urn:entity:video_game": [],\n      "urn:entity:destination": []\n    }\n  }\n}\n```'

In [23]:
def force_json_quotes(text):
    return re.sub(r'(?<=\{|\s)(\w+)(?=\s*:)', r'"\1"', text)

def clean_llm_json_output(llm_output: str) -> dict:
    try:
        logger.info("##### INITIALIZING LLM_PARSER.PY - clean_llm_json_output ##### ")
        cleaned = re.sub(r"```json|```", "", llm_output).strip()
        cleaned = force_json_quotes(cleaned)
        returned = json.loads(cleaned)
        logger.info("##### FINISHED -- INITIALIZING LLM_PARSER.PY - clean_llm_json_output ##### ")
        return returned      
    except Exception as e:
        logger.error(f"ERROR INITIALIZING LLM_PARSER.PY - clean_llm_json_output : {e}")
        raise CustomException("ERROR INITIALIZING LLM_PARSER.PY - clean_llm_json_output", e)
    
def flatten_qloo_payload(payload: dict) -> dict:
        try:
                logger.info("##### INITIALIZING LLM_PARSER.PY - flatten_qloo_payload ##### ")
                params = {
                        "filter.type": payload["filter_type"],
                        "take": payload["take"]
                }

                # Demographics
                demographics = payload.get("signals", {}).get("demographics", {})
                for key, value in demographics.items():
                        params[f"signal.demographics.{key}"] = value

                # Locations
                locations = payload.get("signals", {}).get("location", {}).get("query", [])
                for loc in locations:
                        
                        params.setdefault("signal.location.query", []).append(loc)

                # Entities
                entities = payload.get("signals", {}).get("entities", {})
                for urn_type, values in entities.items():
                       for v in values:
                             params.setdefault(f"signal.entity.{urn_type}", []).append(v)

                logger.info("##### FINISHED -- INITIALIZING LLM_PARSER.PY - flatten_qloo_payload ##### ")
                return params
        except Exception as e:
                logger.error(f"ERROR INITIALIZING LLM PARSER.PY - flatten_qloo_payload : {e}")
                raise CustomException("ERROR INITIALIZING LLM PARSER.PY - flatten_qloo_payload", e)


def is_valid_qloo_payload(payload: dict) -> (bool, str):
    if not isinstance(payload, dict):
        return False, "The response is not a valid JSON object."

    has_location = "signal.location.query" in payload and bool(payload["signal.location.query"])
    has_entities = any(k.startswith("signal.interests.entities") for k in payload)
    has_tags = any(k.startswith("signal.interests.tags") for k in payload)

    if not (has_location or has_entities or has_tags):
        return False, (
            "Please specify at least a location (e.g., 'New York'), "
            "a cultural interest (e.g., a movie, artist, or video game), or a tag."
        )

    return True, ""


In [25]:
payload = clean_llm_json_output(parsed_response)
params = flatten_qloo_payload(payload)
        
is_valid, error_msg = is_valid_qloo_payload(params)

In [11]:
payload

{'filter_type': 'urn:tag',
 'take': 10,
 'signals': {'demographics': {'age': '35_and_younger', 'gender': 'female'},
  'location': {'query': ['New York']},
  'entities': {'urn:entity:artist': [],
   'urn:entity:book': [],
   'urn:entity:movie': [],
   'urn:entity:tv_show': [],
   'urn:entity:video_game': [],
   'urn:entity:destination': []}}}

In [26]:
params

{'filter.type': 'urn:tag',
 'take': 10,
 'signal.demographics.age': '35_and_younger',
 'signal.demographics.gender': 'female',
 'signal.location.query': ['New York']}

In [27]:
is_valid

True

In [28]:
error_msg

''

# Attemp with a normal sentence

In [30]:
normal_sentence = await extract_preferences("I like video games and science fiction movies", "deepseek/deepseek-chat-v3-0324:free", OPENROUTER_API_KEY)

In [40]:
normal_sentence

'```json\n{\n  "filter_type": "urn:tag",\n  "take": 10,\n  "signals": {\n    "demographics": {},\n    "location": {},\n    "entities": {\n      "urn:entity:artist": [],\n      "urn:entity:book": [],\n      "urn:entity:movie": [],\n      "urn:entity:tv_show": [],\n      "urn:entity:video_game": [],\n      "urn:entity:destination": []\n    }\n  }\n}\n```'

In [31]:
normal_payload = clean_llm_json_output(normal_sentence)
normal_params = flatten_qloo_payload(normal_payload)
        
normal_is_valid, normal_error_msg = is_valid_qloo_payload(normal_params)

In [32]:
normal_payload = clean_llm_json_output(normal_sentence)

In [33]:
normal_params

{'filter.type': 'urn:tag', 'take': 10}

In [34]:
normal_is_valid

False

In [35]:
normal_error_msg

"Please specify at least a location (e.g., 'New York'), a cultural interest (e.g., a movie, artist, or video game), or a tag."

# Attemp with instruccions

In [38]:
instruccion_sentence = await extract_preferences("I'm creating a show for Gen Z women in Brooklyn who are into astrology, TikTok trends, indie pop, and surreal coming-of-age stories.", "deepseek/deepseek-chat-v3-0324:free", OPENROUTER_API_KEY)

In [39]:
instruccion_sentence

'```json\n{\n  "filter_type": "urn:tag",\n  "take": 10,\n  "signals": {\n    "demographics": {\n      "age": "35_and_younger",\n      "gender": "female"\n    },\n    "location": {\n      "query": ["Brooklyn"]\n    },\n    "entities": {\n      "urn:entity:artist": ["indie pop"],\n      "urn:entity:book": ["surreal coming-of-age stories"],\n      "urn:entity:movie": [],\n      "urn:entity:tv_show": [],\n      "urn:entity:video_game": [],\n      "urn:entity:destination": []\n    }\n  }\n}\n```'

In [41]:
instruccion_payload = clean_llm_json_output(instruccion_sentence)
instruccion_params = flatten_qloo_payload(instruccion_payload)
        
instruccion_is_valid, instruccion_error_msg = is_valid_qloo_payload(instruccion_params)

In [42]:
instruccion_payload

{'filter_type': 'urn:tag',
 'take': 10,
 'signals': {'demographics': {'age': '35_and_younger', 'gender': 'female'},
  'location': {'query': ['Brooklyn']},
  'entities': {'urn:entity:artist': ['indie pop'],
   'urn:entity:book': ['surreal coming-of-age stories'],
   'urn:entity:movie': [],
   'urn:entity:tv_show': [],
   'urn:entity:video_game': [],
   'urn:entity:destination': []}}}

In [43]:
instruccion_params

{'filter.type': 'urn:tag',
 'take': 10,
 'signal.demographics.age': '35_and_younger',
 'signal.demographics.gender': 'female',
 'signal.location.query': ['Brooklyn'],
 'signal.entity.urn:entity:artist': ['indie pop'],
 'signal.entity.urn:entity:book': ['surreal coming-of-age stories']}

In [44]:
instruccion_is_valid

True

In [45]:
instruccion_error_msg

''

# Qloo try 2


In [13]:
import json
import re

def clean_llm_json_output(llm_output: str) -> dict:
    """
    Limpia un bloque de texto devuelto por un LLM en formato markdown/json
    y lo convierte en un diccionario usable.
    """
    try:
        # Eliminar bloque de markdown como ```json\n...\n```
        cleaned = re.sub(r"```json|```", "", llm_output).strip()
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        raise ValueError(f"❌ Error al decodificar JSON del LLM: {e}")


In [14]:
def flatten_qloo_payload(payload: dict) -> dict:
    params = {
        "filter.type": payload["filter_type"],
        "take": payload["take"]
    }

    # Demographics
    demographics = payload.get("signals", {}).get("demographics", {})
    for key, value in demographics.items():
        params[f"signal.demographics.{key}"] = value

    # Locations
    locations = payload.get("signals", {}).get("location", {}).get("query", [])
    for loc in locations:
        # Qloo acepta múltiples signal.location.query repitiendo la clave
        params.setdefault("signal.location.query", []).append(loc)

    # Entities
    entities = payload.get("signals", {}).get("entities", {})
    for urn_type, values in entities.items():
        for v in values:
            params.setdefault(f"signal.entity.{urn_type}", []).append(v)

    return params


In [15]:
import requests

def call_qloo_insights(params: dict, api_key: str) -> dict:
    url = "https://hackathon.api.qloo.com/v2/insights"
    headers = {
        "x-api-key": api_key
    }

    response = requests.get(url, headers=headers, params=params)
    if response.status_code == 200:
        return response.json()
    else:
        raise ValueError(f"❌ Qloo API Error {response.status_code}: {response.text}")


In [16]:
payload = clean_llm_json_output(parsed_response)

In [17]:
payload

{'filter_type': 'urn:tag',
 'take': 10,
 'signals': {'demographics': {'age': '35_and_younger', 'gender': 'female'},
  'location': {'query': ['New York']},
  'entities': {'urn:entity:artist': [],
   'urn:entity:book': [],
   'urn:entity:movie': [],
   'urn:entity:tv_show': [],
   'urn:entity:video_game': [],
   'urn:entity:destination': []}}}

In [18]:
params = flatten_qloo_payload(payload)

In [19]:
params

{'filter.type': 'urn:tag',
 'take': 10,
 'signal.demographics.age': '35_and_younger',
 'signal.demographics.gender': 'female',
 'signal.location.query': ['New York']}

In [20]:
result = call_qloo_insights(params, QLOO_API_KEY)

In [21]:
result

{'success': True,
 'results': {'tags': [{'tag_id': 'urn:tag:keyword:qloo:reddit',
    'name': 'Reddit',
    'types': ['urn:entity:tv_show', 'urn:entity:podcast', 'urn:entity:movie'],
    'subtype': 'urn:tag:keyword:qloo',
    'tag_value': 'urn:tag:keyword:qloo:reddit',
    'query': {'affinity': 0.9899141643987716}},
   {'tag_id': 'urn:tag:archetype:qloo:family_bond',
    'name': 'Family bond',
    'types': ['urn:entity:tv_show', 'urn:entity:podcast', 'urn:entity:movie'],
    'subtype': 'urn:tag:archetype:qloo',
    'tag_value': 'urn:tag:archetype:qloo:family_bond',
    'query': {'affinity': 0.9741679221367824}},
   {'tag_id': 'urn:tag:subgenre:qloo:heartwarming',
    'name': 'Heartwarming',
    'types': ['urn:entity:tv_show', 'urn:entity:movie'],
    'subtype': 'urn:tag:subgenre:qloo',
    'tag_value': 'urn:tag:subgenre:qloo:heartwarming',
    'query': {'affinity': 0.9732078242936452}},
   {'tag_id': 'urn:tag:plot:qloo:predictable',
    'name': 'Predictable',
    'types': ['urn:entity:

In [17]:
tags = result["results"]["tags"]
tags_json = json.dumps(tags, indent=2)


In [19]:
tags_json

'[\n  {\n    "tag_id": "urn:tag:keyword:qloo:reddit",\n    "name": "Reddit",\n    "types": [\n      "urn:entity:tv_show",\n      "urn:entity:podcast",\n      "urn:entity:movie"\n    ],\n    "subtype": "urn:tag:keyword:qloo",\n    "tag_value": "urn:tag:keyword:qloo:reddit",\n    "query": {\n      "affinity": 0.9899141643987716\n    }\n  },\n  {\n    "tag_id": "urn:tag:archetype:qloo:family_bond",\n    "name": "Family bond",\n    "types": [\n      "urn:entity:tv_show",\n      "urn:entity:podcast",\n      "urn:entity:movie"\n    ],\n    "subtype": "urn:tag:archetype:qloo",\n    "tag_value": "urn:tag:archetype:qloo:family_bond",\n    "query": {\n      "affinity": 0.9741679221367824\n    }\n  },\n  {\n    "tag_id": "urn:tag:subgenre:qloo:heartwarming",\n    "name": "Heartwarming",\n    "types": [\n      "urn:entity:tv_show",\n      "urn:entity:movie"\n    ],\n    "subtype": "urn:tag:subgenre:qloo",\n    "tag_value": "urn:tag:subgenre:qloo:heartwarming",\n    "query": {\n      "affinity": 0.

# QLOO

In [ ]:
import requests
import json
import os


def parse_llm_json_string(llm_response: str) -> dict:
    cleaned = llm_response.strip("`\n ")

    if cleaned.startswith("json"):
        cleaned = cleaned[len("json"):].strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        raise ValueError(f"❌ Error parsing JSON from LLM: {e}")


def call_qloo_from_llm_json(llm_json_str: str) -> dict | None:

    try:
        config = parse_llm_json_string(llm_json_str)
    except ValueError as e:
        print(str(e))
        return None

    # Extraer parámetros desde el JSON
    filter_type = config.get("filter_type", "urn:tag")
    take = config.get("take", 10)
    age = config.get("age")
    locations = config.get("locations", [])

    # Construir query params para la URL
    params = {
        "filter.type": filter_type,
        "take": take
    }

    if age:
        params["signal.demographics.age"] = age

    for loc in locations:
        params.setdefault("signal.location.query", []).append(loc)


    if not QLOO_API_KEY:
        print(" No QLOO_API_KEY found in environment.")
        return None

    headers = {
        "x-api-key": QLOO_API_KEY
    }

    url = "https://hackathon.api.qloo.com/v2/insights"
    response = requests.get(url, headers=headers, params=params)

    if response.status_code == 200:
        return response.json()
    else:
        print(f"Qloo API Error {response.status_code}: {response.text}")
        return None


# Ejemplo de uso (simulando respuesta del LLM)
if __name__ == "__main__":
    llm_response = '''```json
    {
      "filter_type": "urn:tag",
      "take": 10,
      "age": "35_and_younger",
      "locations": ["San Francisco", "Los Angeles"]
    }
    ```'''

    result = call_qloo_from_llm_json(llm_response)

    if result:
        print("✅ Qloo API Result:")
        print(json.dumps(result, indent=2))


✅ Qloo API Result:
{
  "success": true,
  "results": {
    "tags": [
      {
        "tag_id": "urn:tag:keyword:qloo:reddit",
        "name": "Reddit",
        "types": [
          "urn:entity:tv_show",
          "urn:entity:podcast",
          "urn:entity:movie"
        ],
        "subtype": "urn:tag:keyword:qloo",
        "tag_value": "urn:tag:keyword:qloo:reddit",
        "query": {
          "affinity": 0.9898583594966761
        }
      },
      {
        "tag_id": "urn:tag:audience:qloo:internet_savvy",
        "name": "Internet-Savvy",
        "types": [
          "urn:entity:artist",
          "urn:entity:tv_show",
          "urn:entity:podcast",
          "urn:entity:movie"
        ],
        "subtype": "urn:tag:audience:qloo",
        "tag_value": "urn:tag:audience:qloo:internet_savvy",
        "query": {
          "affinity": 0.9804185335232668
        }
      },
      {
        "tag_id": "urn:tag:archetype:qloo:destined_lovers",
        "name": "Destined lovers",
        "t

In [39]:
with open("qloo_output.json", "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print("✅ Resultado guardado en qloo_output.json")


✅ Resultado guardado en qloo_output.json


In [38]:
params

{'filter.type': 'urn:tag',
 'take': 10,
 'signal.demographics.age': '35_and_younger',
 'signal.location.query': ['San Francisco', 'Los Angeles']}